In [ ]:
import heapq
import copy

# 定义重排九宫的目标状态（0表示空格）
# 布局：
# 1 2 3
# 4 5 6
# 7 8 0
TARGET_STATE = [[1, 2, 3], [4, 5, 6], [7, 8, 0]]

# 定义空格的移动方向：上、下、左、右（行偏移，列偏移）
# 对应方向名称：Up, Down, Left, Right
DIRECTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
DIRECTION_NAMES = ['Up', 'Down', 'Left', 'Right']

class PuzzleNode:
    """封装九个宫格中的状态节点，包含状态、代价和路径信息"""
    def __init__(self, state, parent=None, move=None, cost_g=0):
        self.state = state          # 当前八数码的状态
        self.parent = parent        # 父节点，用于回溯路径
        self.move = move            # 从父节点到当前节点的移动方向
        self.cost_g = cost_g        # 实际代价：从初始状态到当前状态的移动步数
        self.cost_h = self.compute_manhattan_distance()  # 启发代价：曼哈顿距离
        self.total_cost = self.cost_g + self.cost_h  # 总代价：f = g + h

    def compute_manhattan_distance(self):
        """计算当前状态到目标状态的曼哈顿距离（启发函数）"""
        manhattan_sum = 0
        for row in range(3):
            for col in range(3):
                value = self.state[row][col]
                if value != 0:  # 忽略空格
                    # 计算该数值在目标状态中的目标位置
                    target_row = (value - 1) // 3
                    target_col = (value - 1) % 3
                    # 累加曼哈顿距离
                    manhattan_sum += abs(row - target_row) + abs(col - target_col)
        return manhattan_sum

    def find_blank_position(self):
        """找到空格（0）在当前状态中的行和列位置"""
        for row_idx in range(3):
            for col_idx in range(3):
                if self.state[row_idx][col_idx] == 0:
                    return row_idx, col_idx

    def create_child_nodes(self):
        """生成当前节点的所有合法子节点（空格移动后的状态）"""
        child_nodes = []
        blank_row, blank_col = self.find_blank_position()

        # 遍历所有移动方向
        for idx, (dr, dc) in enumerate(DIRECTIONS):
            new_row = blank_row + dr
            new_col = blank_col + dc

            # 检查新位置是否在3x3的棋盘范围内
            if 0 <= new_row < 3 and 0 <= new_col < 3:
                # 深拷贝当前状态，避免修改原状态
                new_state = copy.deepcopy(self.state)
                # 交换空格和相邻位置的数值
                new_state[blank_row][blank_col], new_state[new_row][new_col] = \
                    new_state[new_row][new_col], new_state[blank_row][blank_col]
                # 创建子节点，代价g加1
                child_node = PuzzleNode(new_state, self, DIRECTION_NAMES[idx], self.cost_g + 1)
                child_nodes.append(child_node)
        return child_nodes

    def __lt__(self, other):
        """定义小于运算符，用于优先队列的排序（按总代价升序）"""
        return self.total_cost < other.total_cost

    def retrieve_solution_path(self):
        """回溯从初始状态到当前状态的路径"""
        path = []
        current_node = self
        # 从当前节点（目标状态）回溯到初始节点
        while current_node is not None:
            path.append((current_node.move, current_node.state))
            current_node = current_node.parent
        # 反转路径，得到从初始到目标的顺序
        return path[::-1]

def a_star_8_puzzle_solver(initial_state):
    """使用A*算法求解重排九宫问题"""
    # 初始化优先队列（最小堆），存储待扩展的节点
    open_queue = []
    # 将初始状态的节点加入优先队列
    heapq.heappush(open_queue, PuzzleNode(initial_state))

    # 记录已访问的状态
    closed_states = set()
    state_tuple = tuple(tuple(row) for row in initial_state)
    closed_states.add(state_tuple)

    # 统计扩展的节点数量
    expanded_nodes_count = 0

    # 开始A*搜索循环
    while open_queue:
        # 取出总代价最小的节点
        current_node = heapq.heappop(open_queue)
        expanded_nodes_count += 1

        # 判断是否到达目标状态
        if current_node.state == TARGET_STATE:
            # 返回解决方案路径和扩展的节点数
            return current_node.retrieve_solution_path(), expanded_nodes_count

        # 生成当前节点的所有子节点
        for child_node in current_node.create_child_nodes():
            # 将子节点的状态转换为嵌套元组，用于哈希判断
            child_state_tuple = tuple(tuple(row) for row in child_node.state)
            # 如果该状态未被访问过，则加入队列和已访问集合
            if child_state_tuple not in closed_states:
                closed_states.add(child_state_tuple)
                heapq.heappush(open_queue, child_node)

    # 若队列为空，说明该状态无解
    return None, expanded_nodes_count

def display_puzzle_board(state):
    """格式化打印九宫格的棋盘状态（优化输出格式）"""
    for row in state:
        # 打印每行，元素之间加空格，更美观
        print(f"{row[0]} {row[1]} {row[2]}")
    print("====") 

if __name__ == "__main__":
    # 设置八数码的初始状态
    initial_state = [
        [0, 1, 3],
        [4, 2, 6],
        [7, 5, 8]
    ]

    # 打印初始状态
    print("九宫格初始状态：")
    display_puzzle_board(initial_state)

    # 执行A*算法求解
    print("正在执行A*算法求解重排九宫问题...")
    solution_path, nodes_count = a_star_8_puzzle_solver(initial_state)

    # 处理求解结果
    if solution_path:
        print(f"求解成功！共扩展节点数量：{nodes_count}")
        print(f"最优路径的移动步数：{len(solution_path) - 1}")
        print("\n详细的解题步骤：")
        for step_idx, (move, state) in enumerate(solution_path):
            if step_idx == 0:
                # 跳过初始状态（无移动动作）
                continue
            print(f"第 {step_idx} 步：将空格向 {move} 移动")
            display_puzzle_board(state)
    else:
        print("该初始状态无解！")

九宫格初始状态：
0 1 3
4 2 6
7 5 8
====
正在执行A*算法求解重排九宫问题...
求解成功！共扩展节点数量：5
最优路径的移动步数：4

详细的解题步骤：
第 1 步：将空格向 Right 移动
1 0 3
4 2 6
7 5 8
====
第 2 步：将空格向 Down 移动
1 2 3
4 0 6
7 5 8
====
第 3 步：将空格向 Down 移动
1 2 3
4 5 6
7 0 8
====
第 4 步：将空格向 Right 移动
1 2 3
4 5 6
7 8 0
====
